# Day 25: Multi-Cloud Capacity Management
> *100 Days of Inference* | Layer: **Infrastructure** | Book: *Inference Engineering* Ch 7.3 (pp. 193–198)

**Prerequisite:** Day 24


## What problem does this solve?

At scale, GPUs must be distributed across multiple cloud providers and regions. A single cloud has limited capacity, availability zones fail, and end users are global. Multi-cloud inference introduces new challenges: where do requests go, how is heterogeneous compute managed, and what happens when one cloud has an outage?


## Concept Overview

Multi-cloud inference requires:

1. **A global control plane:** Manages model deployment and global scaling decisions
2. **Per-cloud workload planes:** Handle local inference traffic independently
3. **Global load balancer:** Routes user requests to the closest available cluster

Three types of GPU providers:
- **Hyperscalers:** AWS, GCP, Azure — high reliability, premium cost
- **Neoclouds:** CoreWeave, Nebius — GPU-focused, mid-tier cost
- **Resellers:** Secondary markets like SF Compute Company — lowest cost, variable reliability

GPU procurement mechanisms: Reserved (blocks, discounted), On-demand (flexible, expensive), Spot (cheapest, can be pre-empted).


In [1]:
!pip install -q numpy matplotlib torch 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10
GPU Memory: 128.5 GB


## Part 1: GPU Procurement Economics

Cost model for GPU procurement. Reserved instances are 30-60% cheaper than on-demand but require commitment. Spot instances can be 70% cheaper but can be pre-empted with ~2 minute notice.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# GPU cost model
procurement_types = {
    "Reserved (1yr)":   {"multiplier": 0.4, "interruption_prob": 0},
    "Reserved (3mo)":   {"multiplier": 0.6, "interruption_prob": 0},
    "On-demand":        {"multiplier": 1.0, "interruption_prob": 0},
    "Spot":             {"multiplier": 0.3, "interruption_prob": 0.05},  # 5% hourly
}

h100_ondemand_hr = 30  # $30/hr for 8xH100 node

print("GPU Procurement Cost Analysis (8xH100 node, $30/hr on-demand)")
print()
print(f"{'Type':<20} {'Cost/hr':>10} {'Interruption%':>15} {'Effective cost*':>17}")
print("-" * 65)
for name, props in procurement_types.items():
    cost = h100_ondemand_hr * props["multiplier"]
    interr = props["interruption_prob"] * 100
    # Effective cost: account for interruption cost (redeploy + lost work)
    redeploy_cost = props["interruption_prob"] * (0.5 * h100_ondemand_hr)  # 30min redeploy
    effective = cost + redeploy_cost
    print(f"{name:<20} ${cost:>8.2f} {interr:>14.1f}% ${effective:>15.2f}")

print("* Effective cost includes estimated redeploy overhead for interruptions")
print()

# Multi-cloud strategy: baseline reserved + spot/on-demand for spikes
print("Recommended multi-cloud strategy:")
print("  Baseline (70% of traffic): Reserved instances across 2 providers")
print("  Buffer (20% of traffic): On-demand in primary region")
print("  Spike handling (10% of traffic): Spot instances (accept pre-emption risk)")
print()
print("  Benefits:")
print("  - Cost reduction: ~40% vs all on-demand")
print("  - Reliability: 2-provider redundancy")
print("  - Flexibility: on-demand buffer for unexpected spikes")

GPU Procurement Cost Analysis (8xH100 node, $30/hr on-demand)

Type                    Cost/hr   Interruption%   Effective cost*
-----------------------------------------------------------------
Reserved (1yr)       $   12.00            0.0% $          12.00
Reserved (3mo)       $   18.00            0.0% $          18.00
On-demand            $   30.00            0.0% $          30.00
Spot                 $    9.00            5.0% $           9.75
* Effective cost includes estimated redeploy overhead for interruptions

Recommended multi-cloud strategy:
  Baseline (70% of traffic): Reserved instances across 2 providers
  Buffer (20% of traffic): On-demand in primary region
  Spike handling (10% of traffic): Spot instances (accept pre-emption risk)

  Benefits:
  - Cost reduction: ~40% vs all on-demand
  - Reliability: 2-provider redundancy
  - Flexibility: on-demand buffer for unexpected spikes


## Part 2: Geo-Aware Routing

Latency adds 5ms per time zone crossed. Routing Singapore users to San Francisco adds 75ms of pure network latency — often more than the inference latency itself.

In [3]:
import numpy as np

# Network latency model (5ms per time zone)
regions = {
    "us-east-1 (Virginia)":     {"lat": 38.9, "lon": -77.0, "timezone_offset": -5},
    "us-west-2 (Oregon)":       {"lat": 45.5, "lon": -122.9, "timezone_offset": -8},
    "eu-west-1 (Ireland)":      {"lat": 53.3, "lon": -6.3, "timezone_offset": 0},
    "ap-southeast-1 (Singapore)": {"lat": 1.3, "lon": 103.8, "timezone_offset": 8},
    "ap-northeast-1 (Tokyo)":   {"lat": 35.7, "lon": 139.7, "timezone_offset": 9},
}

user_locations = {
    "New York User":    (-74.0, 40.7),
    "London User":      (-0.1, 51.5),
    "Tokyo User":       (139.7, 35.7),
    "Sydney User":      (151.2, -33.9),
    "Singapore User":   (103.8, 1.3),
}

def haversine_km(lon1, lat1, lon2, lat2):
    from math import radians, cos, sin, asin, sqrt
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 2 * 6371 * asin(sqrt(a))

def network_latency_ms(km):
    # Speed of light in fiber: ~2/3 * c, plus routing overhead
    return km / 200e3 * 1000 + 5  # ms

print("Geo-aware routing: latency to each datacenter (ms)")
print()
header = f"{'User':<25} " + " ".join(f"{r[:15]:>16}" for r in regions)
print(header)
print("-" * (25 + 17 * len(regions)))

best_routes = {}
for user, (lon, lat) in user_locations.items():
    row = f"{user:<25}"
    latencies = []
    for region, specs in regions.items():
        km = haversine_km(lon, lat, specs["lon"], specs["lat"])
        latency = network_latency_ms(km)
        latencies.append(latency)
        row += f" {latency:>15.1f}ms"
    print(row)
    best_idx = latencies.index(min(latencies))
    best_routes[user] = list(regions.keys())[best_idx]

print()
print("Optimal region per user:")
for user, region in best_routes.items():
    print(f"  {user}: {region}")

Geo-aware routing: latency to each datacenter (ms)

User                       us-east-1 (Virg  us-west-2 (Oreg  eu-west-1 (Irel  ap-southeast-1   ap-northeast-1 
--------------------------------------------------------------------------------------------------------------
New York User                         6.6ms            24.7ms            30.6ms            81.7ms            59.2ms
London User                          34.5ms            44.6ms             7.3ms            59.2ms            52.8ms
Tokyo User                           59.5ms            43.9ms            52.9ms            31.6ms             5.0ms
Sydney User                          83.6ms            66.6ms            91.1ms            36.5ms            44.2ms
Singapore User                       82.7ms            70.4ms            61.0ms             5.0ms            31.6ms

Optimal region per user:
  New York User: us-east-1 (Virginia)
  London User: eu-west-1 (Ireland)
  Tokyo User: ap-northeast-1 (Tokyo)
  Sydney U

## Try These Experiments

1. **Cost optimizer:** Given a traffic forecast for the next 30 days (peak=200 req/s, average=80 req/s, each requiring 1 GPU), design the optimal mix of Reserved 1yr + On-demand + Spot instances across two clouds to minimize cost while maintaining 99.5% availability.

2. **Failover simulation:** Simulate a partial cloud outage where 30% of capacity in us-east-1 goes down. Measure how quickly a geo-aware load balancer redirects traffic and the latency impact for us-east users during the failover window.

3. **VRAM budget across clouds:** Three models (70B, 13B, 7B) need to be served with different traffic levels. Allocate GPUs across three cloud regions to minimize cost while meeting per-model, per-region latency Service Level Agreements (SLAs).


## Key Takeaways

- Multi-cloud requires a global control plane + per-cloud workload planes — two separate concerns.
- Hyperscalers = reliability; neoclouds = GPU-focused at better price; spot = cheap but pre-emptible.
- Geo-aware routing saves 50–150 ms of network latency by routing users to the nearest datacenter.
- Reserved baseline + on-demand buffer + spot peaks = optimal cost/reliability tradeoff.
- Llama 3 reported ~1 hardware failure per 50,000 GPU-hours — at 8 GPUs running for a year (≈70,000 GPU-hours), failure is expected, not exceptional.
- **What's next:** Day 26 — Zero-downtime deployment and cost estimation.

## References
- *Inference Engineering* Ch 7.3 (pp. 193–198)
- Grattafiori et al., *The Llama 3 Herd of Models* (2024) — referenced for the 419-interruption / 16,000-GPU figure
